# From the Perceptron to Backpropagation

This notebook implements four landmark neural-network learners in historical order.

1. **Part 1, Rosenblatt's perceptron (1958):** a threshold unit trained by an error-correction rule.
2. **Part 2, Widrow & Hoff's ADALINE (1960):** a linear unit trained by the LMS (delta) rule.
3. **Part 3, Widrow's MADALINE (early 1960s):** several ADALINEs with hard thresholds feeding a fixed logic gate, trained by Madaline Rule I.
4. **Part 4, the multilayer perceptron with backpropagation (1974 / 1986):** differentiable hidden units, with every weight trained by an exact gradient.

The two single units are traced step by step on **AND**, and both fail on **XOR**. Their real difference shows up on noisy, overlapping data, where only ADALINE settles down. MADALINE is the first step past the single-unit limit: it solves XOR with a heuristic. Part 4 solves it with gradients, and checks those gradients numerically.

# Part 1: The 1958 Perceptron (Rosenblatt)

## Architecture

Rosenblatt's perceptron had three layers. Only the A → R weights are learned.

```
 Retina (S-units)      Association (A-units)        Response (R-unit)
 sensory, binary       fixed, random connections    threshold unit

   [S1] ----\          /--> [A1] --w1--\
   [S2] -----+--------+---> [A2] --w2---+--->  [ R ] ---> 1 / 0
   [S3] -----+--------+---> [A3] --w3---+
   [S4] ----/          \--> [An] --wn--/
                                  ^
                     only these weights are learned
```

In the worked examples below the A-layer is the identity: the inputs go straight to R. That is the simplest case and is all AND needs.

## The learning rule

Let `x` be the 0/1 input vector, `t` the target, `y` the actual response.

1. `net = w·x + b`, and `y = 1` if `net > 0`, else `y = 0`.
2. If `y == t`: do nothing.
3. If `y = 0` but `t = 1`: add 1 to the weight of every active input (`w ← w + x`, `b ← b + 1`).
4. If `y = 1` but `t = 0`: subtract 1 from the weight of every active input (`w ← w − x`, `b ← b − 1`).
5. Repeat passes over the training set until a whole pass makes no errors.

Both update cases are the single formula `w ← w + η (t − y) x`, with the bias treated as a weight on a constant input of 1.

**Historical note.** Rosenblatt's 1958 paper described the reinforcement scheme in prose. The compact formula comes from his 1962 book *Principles of Neurodynamics* and later treatments. It is the same rule.

**Guarantee (Rosenblatt / Novikoff):** if the data are linearly separable, this procedure reaches zero errors after finitely many updates.

In [1]:
import numpy as np


def predict(w, b, x):
    """Threshold unit: fires (1) if w.x + b > 0, else 0."""
    return 1 if np.dot(w, x) + b > 0 else 0


def train(X, T, eta=1, max_epochs=100, verbose=True):
    """Perceptron error-correction learning.

    Trains until a full pass over the data has zero errors, or max_epochs is reached.
    Returns (w, b, converged, epochs_used, history), where history is a list of
    (errors_in_epoch, weights_at_end_of_epoch, bias_at_end_of_epoch).
    """
    X = np.asarray(X)
    w = np.zeros(X.shape[1], dtype=int)
    b = 0
    history = []

    if verbose:
        print(f"{'ep':>2} {'input':>7} {'net':>4} {'y':>2} {'t':>2}  {'action':<9} (w1, w2, b)")

    for epoch in range(1, max_epochs + 1):
        errors = 0
        for x, t in zip(X, T):
            net = (np.dot(w, x) + b).item()
            y = 1 if net > 0 else 0
            if y == t:
                action = "ok"
            else:
                errors += 1
                action = "add" if t > y else "subtract"
                w = w + eta * (t - y) * x
                b = b + eta * (t - y)
            if verbose:
                print(f"{epoch:>2} {str(tuple(x.tolist())):>7} {net:>4g} {y:>2} {t:>2}  "
                      f"{action:<9} ({', '.join(f'{v:g}' for v in w)}, {b:g})")
        history.append((errors, tuple(w.tolist()), np.asarray(b).item()))
        if errors == 0:
            return w, b, True, epoch, history
    return w, b, False, max_epochs, history

## Worked example: learning AND

Start from `w = (0, 0)`, `b = 0`, step size 1. Training set: (0,0)→0, (0,1)→0, (1,0)→0, (1,1)→1.

Every weight update is printed, so you can follow the whole trace.

In [2]:
X = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
T_and = np.array([0, 0, 0, 1])

w, b, converged, epochs, hist = train(X, T_and)
print()
print(f"converged={converged} after {epochs} epochs: w = {w.tolist()}, b = {b}")

ep   input  net  y  t  action    (w1, w2, b)
 1  (0, 0)    0  0  0  ok        (0, 0, 0)
 1  (0, 1)    0  0  0  ok        (0, 0, 0)
 1  (1, 0)    0  0  0  ok        (0, 0, 0)
 1  (1, 1)    0  0  1  add       (1, 1, 1)
 2  (0, 0)    1  1  0  subtract  (1, 1, 0)
 2  (0, 1)    1  1  0  subtract  (1, 0, -1)
 2  (1, 0)    0  0  0  ok        (1, 0, -1)
 2  (1, 1)    0  0  1  add       (2, 1, 0)
 3  (0, 0)    0  0  0  ok        (2, 1, 0)
 3  (0, 1)    1  1  0  subtract  (2, 0, -1)
 3  (1, 0)    1  1  0  subtract  (1, 0, -2)
 3  (1, 1)   -1  0  1  add       (2, 1, -1)
 4  (0, 0)   -1  0  0  ok        (2, 1, -1)
 4  (0, 1)    0  0  0  ok        (2, 1, -1)
 4  (1, 0)    1  1  0  subtract  (1, 1, -2)
 4  (1, 1)    0  0  1  add       (2, 2, -1)
 5  (0, 0)   -1  0  0  ok        (2, 2, -1)
 5  (0, 1)    1  1  0  subtract  (2, 1, -2)
 5  (1, 0)    0  0  0  ok        (2, 1, -2)
 5  (1, 1)    1  1  1  ok        (2, 1, -2)
 6  (0, 0)   -2  0  0  ok        (2, 1, -2)
 6  (0, 1)   -1  0  0  ok        (2, 1

The last epoch is the clean pass with zero errors, so the weights stop changing. Check the learned unit against the truth table, and read off the decision rule.

In [3]:
print(f"decision rule: fire iff {w[0]}*x1 + {w[1]}*x2 + ({b}) > 0\n")
print(" x1 x2 | net | y | t")
print("-------+-----+---+---")
for x, t in zip(X, T_and):
    net = int(np.dot(w, x) + b)
    print(f"  {x[0]}  {x[1]} | {net:>3} | {predict(w, b, x)} | {t}")

decision rule: fire iff 2*x1 + 1*x2 + (-2) > 0

 x1 x2 | net | y | t
-------+-----+---+---
  0  0 |  -2 | 0 | 0
  0  1 |  -1 | 0 | 0
  1  0 |   0 | 0 | 0
  1  1 |   1 | 1 | 1


The boundary is the line `2*x1 + x2 = 2`. Only the point (1,1) lands on the firing side.

```
 x2
  1 |  o           * (1,1) -> 1
    |    \
    |      \   boundary: 2*x1 + x2 = 2
  0 |  o     \  o
    +----------\------ x1
       0         1
```

## The limit: XOR

XOR needs (0,1) and (1,0) to fire while (0,0) and (1,1) stay silent. No single line separates those, so the update loop can never reach a zero-error pass. Instead the weights fall into a repeating cycle (the perceptron cycling theorem). In the run below, every epoch makes 4 errors and the weights at the end of each epoch are identical, so the updates inside an epoch cancel each other out and training goes nowhere.

Minsky and Papert formalized this limitation in 1969.

In [4]:
T_xor = np.array([0, 1, 1, 0])

w, b, converged, epochs, hist = train(X, T_xor, max_epochs=20, verbose=False)
print(f"converged={converged} after {epochs} epochs")
print("errors per epoch:", [h[0] for h in hist])

states = [(h[1], h[2]) for h in hist]
print(f"distinct (w, b) states at epoch ends: {len(set(states))} out of {len(states)} epochs")
print("last 6 epoch-end states (w, b):")
for s in states[-6:]:
    print("  ", s)

converged=False after 20 epochs
errors per epoch: [2, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]
distinct (w, b) states at epoch ends: 2 out of 20 epochs
last 6 epoch-end states (w, b):
   ((-1, 0), 1)
   ((-1, 0), 1)
   ((-1, 0), 1)
   ((-1, 0), 1)
   ((-1, 0), 1)
   ((-1, 0), 1)


## Other Boolean functions

The same code, unchanged, on every two-input Boolean function that has a line separating its 1s from its 0s, plus the two that do not (XOR and XNOR).

In [5]:
functions = {
    "AND":  [0, 0, 0, 1],
    "OR":   [0, 1, 1, 1],
    "NAND": [1, 1, 1, 0],
    "NOR":  [1, 0, 0, 0],
    "XOR":  [0, 1, 1, 0],
    "XNOR": [1, 0, 0, 1],
}

print(f"{'function':<9}{'converged':<11}{'epochs':<8}{'w':<12}b")
for name, T in functions.items():
    w, b, ok, ep, _ = train(X, np.array(T), max_epochs=100, verbose=False)
    print(f"{name:<9}{str(ok):<11}{ep:<8}{str(w.tolist()):<12}{b}")

function converged  epochs  w           b
AND      True       6       [2, 1]      -2
OR       True       4       [1, 1]      0
NAND     True       6       [-2, -1]    3
NOR      True       4       [-1, -1]    1
XOR      False      100     [-1, 0]     1
XNOR     False      100     [1, 0]      0


# Part 2: ADALINE (Widrow & Hoff, 1960)

ADALINE ("ADAptive LInear NEuron") appeared two years after the perceptron. It looks almost identical on the outside, but it changes **where the error is measured**.

- Inputs and targets are **bipolar** (−1 / +1), the original ADALINE convention.
- The weights are corrected using the **linear output** `s = w·x + b`, taken *before* the sign quantizer.
- The error `t − s` is a real number, so **every** sample produces a graded correction, not just the misclassified ones.
- The rule is stochastic gradient descent on squared error. It is called LMS (least mean squares), the Widrow–Hoff rule, or the delta rule.

```
 x1 ---w1--\
 x2 ---w2---+--> [ sum ] --s--> [ quantizer: sign ] --> y  (+1 / -1)
  1 ---b---/        |
                    +-- error = t - s   <-- LMS learns from this
                                              (the perceptron learns from t - y)
```

## The learning rule

For each training pair `(x, t)` with `t ∈ {−1, +1}`:

1. `s = w·x + b` (linear output)
2. `err = t − s`
3. `w ← w + η · err · x`, and `b ← b + η · err`

The quantizer `y = +1 if s ≥ 0 else −1` is used only to classify. It plays no part in learning.

**Why this works.** The gradient of `½ (t − s)²` with respect to `w` is `−(t − s) x`, so each update is a small step downhill on the squared error. (Widrow's papers write the step as `2μ`; here the 2 is absorbed into `η`.)

**Step size.** `η` must be small enough for stability. Widrow's normalized form (α-LMS) uses `η = α / |x|²` with `0 < α < 2`. For 2-D bipolar inputs plus a bias, `|x|² = 3`, so anything well below `2/3` is safe. We use `η = 0.1`.

In [6]:
def adaline_predict(w, b, x):
    """Quantizer on top of the linear output: +1 if s >= 0 else -1."""
    return 1 if np.dot(w, x) + b >= 0 else -1


def adaline_train(X, T, eta=0.1, epochs=100, shuffle=False, seed=0, trace_epochs=0):
    """Widrow-Hoff LMS (delta rule), online: one update per training sample.

    X: (n, d) array. T: targets in {-1, +1}.
    The error signal is err = t - s, where s = w.x + b is the LINEAR output.
    Returns (w, b, history); history[k] = (mse, n_misclassified), measured on the
    whole data set at the end of epoch k+1.
    (trace_epochs prints each update for the first few epochs; 2-D bipolar inputs only.)
    """
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=float)
    rng = np.random.default_rng(seed)
    w = np.zeros(X.shape[1])
    b = 0.0
    history = []

    if trace_epochs:
        print(f"{'ep':>2} {'input':>9} {'s':>7} {'t':>3} {'err':>7}   (w1, w2, b)")

    for epoch in range(1, epochs + 1):
        order = rng.permutation(len(X)) if shuffle else np.arange(len(X))
        for i in order:
            x, t = X[i], T[i]
            s = float(np.dot(w, x) + b)
            err = t - s
            w = w + eta * err * x
            b = b + eta * err
            if epoch <= trace_epochs:
                print(f"{epoch:>2} {f'({x[0]:+.0f},{x[1]:+.0f})':>9} {s:>7.3f} {int(t):>3} {err:>7.3f}   "
                      f"({w[0]:.3f}, {w[1]:.3f}, {b:.3f})")
        s_all = X @ w + b
        mse = float(np.mean((T - s_all) ** 2))
        wrong = int(np.sum(np.where(s_all >= 0, 1, -1) != T))
        history.append((mse, wrong))
    return w, b, history

## Worked example: learning AND (bipolar)

Same problem as Part 1, re-encoded from 0/1 to −1/+1. First the update trace for the first two epochs. Note that **every** row changes the weights, even when the sign of `s` is already right. Then the long run.

In [7]:
Xb = 2 * X - 1                 # 0/1 -> -1/+1
T_and_b = 2 * T_and - 1

print("First two epochs (eta = 0.1):")
adaline_train(Xb, T_and_b, eta=0.1, epochs=2, trace_epochs=2)

w_a, b_a, hist_a = adaline_train(Xb, T_and_b, eta=0.1, epochs=200)
print(f"\nafter 200 epochs: w = {np.round(w_a, 3).tolist()}, b = {b_a:.3f}")
for k in (1, 2, 5, 10, 25, 50, 100, 200):
    mse, wrong = hist_a[k - 1]
    print(f"  epoch {k:>3}: MSE = {mse:.4f}, misclassified = {wrong}")

First two epochs (eta = 0.1):
ep     input       s   t     err   (w1, w2, b)
 1   (-1,-1)   0.000  -1  -1.000   (0.100, 0.100, -0.100)
 1   (-1,+1)  -0.100  -1  -0.900   (0.190, 0.010, -0.190)
 1   (+1,-1)  -0.010  -1  -0.990   (0.091, 0.109, -0.289)
 1   (+1,+1)  -0.089   1   1.089   (0.200, 0.218, -0.180)
 2   (-1,-1)  -0.598  -1  -0.402   (0.240, 0.258, -0.220)
 2   (-1,+1)  -0.202  -1  -0.798   (0.320, 0.178, -0.300)
 2   (+1,-1)  -0.159  -1  -0.841   (0.236, 0.262, -0.384)
 2   (+1,+1)   0.114   1   0.886   (0.324, 0.351, -0.296)

after 200 epochs: w = [0.529, 0.559], b = -0.500
  epoch   1: MSE = 0.5220, misclassified = 0
  epoch   2: MSE = 0.3448, misclassified = 0
  epoch   5: MSE = 0.2533, misclassified = 0
  epoch  10: MSE = 0.2537, misclassified = 0
  epoch  25: MSE = 0.2543, misclassified = 0
  epoch  50: MSE = 0.2543, misclassified = 0
  epoch 100: MSE = 0.2543, misclassified = 0
  epoch 200: MSE = 0.2543, misclassified = 0


LMS is descending toward the **least-squares solution**, which can be computed directly. Compare the two, and check the final unit against the truth table.

In [8]:
A = np.hstack([Xb, np.ones((len(Xb), 1))])           # augment with a constant 1 for the bias
sol, *_ = np.linalg.lstsq(A, T_and_b, rcond=None)
print("least-squares solution: w =", np.round(sol[:2], 3).tolist(), "  b =", round(float(sol[2]), 3))
print("LMS after 200 epochs:   w =", np.round(w_a, 3).tolist(), "  b =", round(float(b_a), 3))

print("\n x1 x2 |    s    | y | t")
print("-------+---------+---+---")
for x, t in zip(Xb, T_and_b):
    s = float(np.dot(w_a, x) + b_a)
    print(f" {x[0]:+d} {x[1]:+d} | {s:+.3f} | {adaline_predict(w_a, b_a, x):+d} | {t:+d}")

least-squares solution: w = [0.5, 0.5]   b = -0.5
LMS after 200 epochs:   w = [0.529, 0.559]   b = -0.5

 x1 x2 |    s    | y | t
-------+---------+---+---
 -1 -1 | -1.588 | -1 | -1
 -1 +1 | -0.471 | -1 | -1
 +1 -1 | -0.529 | -1 | -1
 +1 +1 | +0.588 | +1 | +1


ADALINE classifies all four points correctly, yet the **MSE does not go to zero**. A linear output cannot equal exactly ±1 on all four AND points at once, so the error signal never vanishes and the weights keep getting nudged toward the best compromise. The perceptron is different: once every point is classified correctly it stops updating entirely.

The MSE settles at about 0.254, just above the least-squares minimum of 0.25. With a constant step size, online LMS hovers in a small neighborhood of the optimum instead of landing exactly on it, which is why the learned weights differ slightly from the least-squares ones.

## XOR again

ADALINE is still a single linear unit, so it cannot represent XOR either. The least-squares fit to XOR is the all-zero unit, which predicts 0 everywhere. Online LMS with a constant step ends up jittering around that point, so the leftover outputs are tiny and the quantized predictions are essentially arbitrary.

In [9]:
T_xor_b = 2 * T_xor - 1
w_x, b_x, hist_x = adaline_train(Xb, T_xor_b, eta=0.1, epochs=200)

print(f"w = {np.round(w_x, 3).tolist()}, b = {b_x:.3f}")
print(f"MSE after 200 epochs: {hist_x[-1][0]:.4f}   (predicting 0 everywhere gives 1.0)")
print(f"misclassified: {hist_x[-1][1]} of 4")
print("linear outputs s:", np.round(Xb @ w_x + b_x, 3).tolist())

w = [-0.059, -0.118], b = -0.000
MSE after 200 epochs: 1.0173   (predicting 0 everywhere gives 1.0)
misclassified: 2 of 4
linear outputs s: [0.176, -0.059, 0.059, -0.176]


## Where ADALINE differs: overlapping classes

On linearly separable data both learners can win. The difference appears when the classes **overlap**, as real data usually does. The perceptron rule takes a full-size step on every mistake, and on overlapping data the mistakes never stop, so its weights keep jumping. ADALINE's steps are scaled by the error and by a small `η`, so its weights hover in a small neighborhood of the least-squares solution.

Two Gaussian blobs, centred at (+1, +1) and (−1, −1), each with standard deviation 1, so no line separates them perfectly. The points are mixed into one fixed random order, and **both learners see exactly the same order every epoch**. The measure is the same for both: how many of the 200 points the current weights misclassify at the end of each epoch.

In [10]:
rng = np.random.default_rng(42)
n = 100
pos = rng.normal(loc=[+1.0, +1.0], scale=1.0, size=(n, 2))
neg = rng.normal(loc=[-1.0, -1.0], scale=1.0, size=(n, 2))
Xn = np.vstack([pos, neg])
Tn_01 = np.array([1] * n + [0] * n)      # perceptron targets
perm = rng.permutation(2 * n)            # mix the classes once; both learners use this fixed order
Xn, Tn_01 = Xn[perm], Tn_01[perm]
Tn_pm = 2 * Tn_01 - 1                    # ADALINE targets
N = len(Xn)


def n_wrong_01(w, b):
    """Points misclassified by a perceptron-style unit (fires if w.x + b > 0)."""
    return int(np.sum((Xn @ w + b > 0).astype(int) != Tn_01))


# --- Perceptron, reusing train() from Part 1; history holds the weights at the end of each epoch ---
_, _, ok, _, hist_p = train(Xn, Tn_01, eta=1, max_epochs=100, verbose=False)
wrong_p = [n_wrong_01(np.array(h[1]), h[2]) for h in hist_p]
print(f"PERCEPTRON  converged={ok}")
print("  misclassified at the end of each epoch, epochs 1-15:", wrong_p[:15])
print(f"  epochs 51-100: min {min(wrong_p[50:])}, max {max(wrong_p[50:])}, last {wrong_p[-1]}  (of {N})")

# --- ADALINE, same data and same order ---
w_n, b_n, hist_n = adaline_train(Xn, Tn_pm, eta=0.01, epochs=100)
wrong_n = [h[1] for h in hist_n]
print("\nADALINE (eta = 0.01)")
print("  misclassified at the end of each epoch, epochs 1-15:", wrong_n[:15])
print(f"  epochs 51-100: min {min(wrong_n[50:])}, max {max(wrong_n[50:])}, last {wrong_n[-1]}  (of {N})")
print("  MSE at epochs 1, 5, 10, 25, 50, 100:", [round(hist_n[k - 1][0], 4) for k in (1, 5, 10, 25, 50, 100)])

# --- Least-squares optimum for reference ---
A_n = np.hstack([Xn, np.ones((len(Xn), 1))])
sol_n, *_ = np.linalg.lstsq(A_n, Tn_pm, rcond=None)
s_ls = A_n @ sol_n
print("\nLEAST-SQUARES optimum")
print(f"  MSE = {np.mean((Tn_pm - s_ls) ** 2):.4f}, "
      f"misclassified = {int(np.sum(np.where(s_ls >= 0, 1, -1) != Tn_pm))} of {2 * n}")
print("  weights (w1, w2, b):", np.round(sol_n, 3).tolist())
print("  LMS weights        :", np.round([w_n[0], w_n[1], b_n], 3).tolist())

PERCEPTRON  converged=False
  misclassified at the end of each epoch, epochs 1-15: [18, 17, 19, 12, 19, 17, 15, 19, 14, 26, 14, 15, 19, 14, 26]
  epochs 51-100: min 12, max 27, last 17  (of 200)

ADALINE (eta = 0.01)
  misclassified at the end of each epoch, epochs 1-15: [14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14, 14]
  epochs 51-100: min 14, max 14, last 14  (of 200)
  MSE at epochs 1, 5, 10, 25, 50, 100: [0.3353, 0.3351, 0.3351, 0.3351, 0.3351, 0.3351]

LEAST-SQUARES optimum
  MSE = 0.3333, misclassified = 14 of 200
  weights (w1, w2, b): [0.31, 0.374, 0.005]
  LMS weights        : [0.329, 0.389, 0.015]


**Reading the output.** The perceptron's mistake count swings from epoch to epoch and never stops moving. ADALINE holds steady at the same count as the exact least-squares fit, and its weights sit close to the least-squares weights.

The perceptron's best epochs can even beat ADALINE's count, because least squares minimizes squared error, not the number of mistakes. What ADALINE gives you is stability: a well-defined answer you can stop at, rather than a unit that keeps changing its mind.

## Perceptron vs ADALINE at a glance

| | Perceptron (1958) | ADALINE (1960) |
|---|---|---|
| Output used for learning | thresholded `y` | linear `s`, before the quantizer |
| Error signal | `t − y`, only −1, 0 or +1 | `t − s`, any real number |
| Updates on | misclassified samples only | every sample |
| Encoding | 0 / 1 | −1 / +1 |
| Separable data | reaches zero errors in finitely many updates | approaches the least-squares fit; the MSE stays above zero, and that fit is not guaranteed to separate every separable set |
| Overlapping data | never settles | settles near the least-squares solution |
| XOR | fails | fails |

# Part 3: MADALINE (Widrow, early 1960s)

A single perceptron or ADALINE draws **one** line, so XOR is out of reach. MADALINE ("Many ADALINEs") gets past that by combining several of them.

## Architecture (Madaline I)

- A first layer of adaptive ADALINEs, each with a hard-limiting (sign) output.
- A **fixed** second layer: a logic gate over the hidden outputs, either AND, OR or majority vote.
- Only the first-layer weights are trained. The output gate is wired in and never changes.

```
        +--> [ADALINE 1] --s1--> sign --y1--+
 x -----+                                    +--> [ fixed gate: AND / OR / MAJ ] --> y
        +--> [ADALINE 2] --s2--> sign --y2--+
                 ^
           only these weights are trained
```

**Why this can do XOR.** `XOR(x1, x2) = OR(x1, x2) AND NAND(x1, x2)`. OR and NAND are each linearly separable, so one hidden ADALINE can learn each of them, and a fixed AND gate combines the two.

## The learning rule: Madaline Rule I (MRI)

A hard `sign()` has no useful slope, so there is no error to pass back through it and MRI does not use a gradient. It works by trial and error under the **minimal disturbance principle**: fix the output by changing as few units as possible, and change each one as little as possible.

For each training pattern `(x, t)` with `t ∈ {−1, +1}`:

1. Compute every hidden linear output `s_j = w_j·x + b_j`, the signs `y_j`, and the gate output `y`.
2. If `y == t`: do nothing.
3. Otherwise take the hidden units that currently disagree with the target (`y_j ≠ t`) and rank them by `|s_j|`, smallest first. The unit closest to zero is the cheapest one to flip.
4. Go down that ranking, flipping one more unit at a time, **only until the gate output equals `t`**. Those units are the ones to adapt.
5. Adapt each chosen unit with α-LMS so its linear output moves to the target `t`: `w_j ← w_j + α (t − s_j) x / (|x|² + 1)`, where the `+ 1` is the bias input. With `α = 1` the output lands exactly on `t`, so the sign reverses at once.
6. Present the patterns in a fresh random order each epoch, and stop after a pass with no errors.

Step 3 is safe because AND, OR and majority are *monotone*: pushing a unit toward `t` can never push the gate output away from `t`.

**History.** Madaline I and this rule come from Widrow's group in the early 1960s. Its successor, **MRII** (Winter & Widrow, late 1980s), trains multilayer networks in which *every* layer adapts, trying trial flips of single units and then of pairs or triples. MRII is not implemented here.

In [11]:
def madaline_logic(y, kind):
    """Fixed output gate over the hidden outputs y (each +1 / -1)."""
    if kind == "and":
        return 1 if np.all(y == 1) else -1
    if kind == "or":
        return 1 if np.any(y == 1) else -1
    if kind == "majority":
        return 1 if y.sum() > 0 else -1
    raise ValueError(f"unknown gate: {kind}")


def madaline_predict(W, B, x, out="and"):
    """Hidden ADALINEs (the sign of each linear output), then the fixed gate."""
    y = np.where(W @ x + B >= 0, 1, -1)
    return madaline_logic(y, out)


def madaline_train(X, T, k=2, out="and", alpha=1.0, max_epochs=200, seed=0, init=0.5, trace=False):
    """Madaline Rule I: k adaptive hidden ADALINEs + a fixed output gate.

    X: (n, d) array of -1/+1 inputs. T: targets in {-1, +1}.
    Returns (W, B, converged, epochs_used, history), where history[e] is the number of
    misclassified patterns at the end of epoch e+1.
    (trace=True prints every pattern of every epoch; 2-D inputs only.)
    """
    rng = np.random.default_rng(seed)
    X = np.asarray(X, dtype=float)
    T = np.asarray(T, dtype=int)
    W = rng.uniform(-init, init, size=(k, X.shape[1]))      # small random start breaks symmetry
    B = rng.uniform(-init, init, size=k)
    history = []

    if trace:
        print(f"{'ep':>2} {'input':>9} {'s (hidden)':>16} {'y (hidden)':>11} {'gate':>5} {'t':>3}  action")

    for epoch in range(1, max_epochs + 1):
        for i in rng.permutation(len(X)):
            x, t = X[i], T[i]
            s = W @ x + B
            y = np.where(s >= 0, 1, -1)
            gate = madaline_logic(y, out)

            action = "ok"
            if gate != t:
                # minimal disturbance: wrong units first, least confident (smallest |s|) first
                candidates = [j for j in np.argsort(np.abs(s)) if y[j] != t]
                flipped, y_try = [], y.copy()
                for j in candidates:
                    y_try[j] = t
                    flipped.append(int(j))
                    if madaline_logic(y_try, out) == t:
                        break
                for j in flipped:                           # alpha-LMS toward t, chosen units only
                    step = alpha * (t - s[j]) / (x @ x + 1)
                    W[j] += step * x
                    B[j] += step
                action = "flip " + ",".join(f"h{j + 1}" for j in sorted(flipped))

            if trace:
                xs = f"({x[0]:+.0f},{x[1]:+.0f})"
                print(f"{epoch:>2} {xs:>9} {np.array2string(s, precision=2, floatmode='fixed'):>16} "
                      f"{str(y.tolist()):>11} {gate:>+5d} {t:>+3d}  {action}")

        S = X @ W.T + B
        wrong = sum(madaline_logic(np.where(S[n] >= 0, 1, -1), out) != T[n] for n in range(len(X)))
        history.append(int(wrong))
        if wrong == 0:
            return W, B, True, epoch, history
    return W, B, False, max_epochs, history

## Worked example: learning XOR

Two hidden ADALINEs and a fixed AND gate, on the same bipolar XOR data as Part 2 (targets −1, +1, +1, −1). `seed=4` fixes the random start and the pattern order, so the trace is reproducible.

Each row shows the hidden linear outputs `s` *before* any update. Rows marked `ok` change nothing. On a mistake, only the least-confident wrong hidden unit or units are adapted.

In [12]:
W_m, B_m, ok, ep, hist_m = madaline_train(Xb, T_xor_b, k=2, out="and", seed=4, trace=True)
print(f"\nconverged={ok} after {ep} epochs; misclassified patterns at the end of each epoch: {hist_m}")

ep     input       s (hidden)  y (hidden)  gate   t  action
 1   (-1,+1)    [-0.32 -1.02]    [-1, -1]    -1  +1  flip h1,h2
 1   (-1,-1)      [0.09 0.49]      [1, 1]    +1  -1  flip h1
 1   (+1,-1)    [-0.27  0.10]     [-1, 1]    -1  +1  flip h1
 1   (+1,+1)      [1.79 0.61]      [1, 1]    +1  -1  flip h2
 2   (+1,+1)    [ 1.79 -1.00]     [1, -1]    -1  -1  ok
 2   (-1,-1)    [-0.58  1.03]     [-1, 1]    -1  -1  ok
 2   (-1,+1)      [0.21 0.46]      [1, 1]    +1  +1  ok
 2   (+1,-1)    [ 1.00 -0.44]     [1, -1]    -1  +1  flip h2
 3   (+1,+1)    [ 1.79 -0.52]     [1, -1]    -1  -1  ok
 3   (-1,+1)    [ 0.21 -0.01]     [1, -1]    -1  +1  flip h2
 3   (+1,-1)      [1.00 0.66]      [1, 1]    +1  +1  ok
 3   (-1,-1)    [-0.58  1.84]     [-1, 1]    -1  -1  ok

converged=True after 3 epochs; misclassified patterns at the end of each epoch: [1, 1, 0]


Now look at what the two hidden units learned, and check the whole network against the XOR truth table. Each hidden unit should have become one of the two linearly separable pieces, OR and NAND.

In [13]:
BOOLEAN_NAMES = {          # unit outputs on (-1,-1), (-1,+1), (+1,-1), (+1,+1)
    (-1, -1, -1, +1): "AND", (-1, +1, +1, +1): "OR",
    (+1, +1, +1, -1): "NAND", (+1, -1, -1, -1): "NOR",
}

print("learned hidden units (fire +1 iff s >= 0):")
for j in range(2):
    pattern = tuple(int(v) for v in np.where(Xb @ W_m[j] + B_m[j] >= 0, 1, -1))
    print(f"  h{j + 1}: s = {W_m[j, 0]:+.2f}*x1 {W_m[j, 1]:+.2f}*x2 {B_m[j]:+.2f}   -> {BOOLEAN_NAMES.get(pattern, 'other')}")

print("\n x1 x2 | h1 h2 | gate | t")
print("-------+-------+------+---")
for x, t in zip(Xb, T_xor_b):
    y = np.where(W_m @ x + B_m >= 0, 1, -1)
    print(f" {x[0]:+d} {x[1]:+d} | {y[0]:+d} {y[1]:+d} |  {madaline_predict(W_m, B_m, x):+d}  | {t:+d}")

learned hidden units (fire +1 iff s >= 0):
  h1: s = +0.79*x1 +0.40*x2 +0.61   -> OR
  h2: s = -0.59*x1 -0.42*x2 +0.83   -> NAND

 x1 x2 | h1 h2 | gate | t
-------+-------+------+---
 -1 -1 | -1 +1 |  -1  | -1
 -1 +1 | +1 +1 |  +1  | +1
 +1 -1 | +1 +1 |  +1  | +1
 +1 +1 | +1 -1 |  -1  | -1


## How reliable is MRI?

MRI is a heuristic. Unlike the perceptron rule it has no general convergence theorem, so one lucky run proves little. The cell below trains from 100 different random starts on two problems: XOR, and 3-bit parity (the target is +1 when an even number of the three inputs are −1), which no single-line learner can solve either.

"Solved" is checked independently of the training loop, by re-evaluating the trained network on every pattern.

In [14]:
import itertools

X3 = np.array(list(itertools.product([-1, 1], repeat=3)))
T3 = X3.prod(axis=1)                 # 3-bit parity: +1 when an even number of inputs are -1

experiments = [
    ("XOR",      Xb, T_xor_b, 2, "and"),
    ("XOR",      Xb, T_xor_b, 2, "or"),
    ("XOR",      Xb, T_xor_b, 3, "majority"),
    ("parity-3", X3, T3,      3, "majority"),
    ("parity-3", X3, T3,      4, "and"),
]

print(f"{'problem':<10}{'hidden':<8}{'gate':<10}{'solved':<10}epochs to converge (median / max)")
for name, XX, TT, k, gate in experiments:
    runs = [madaline_train(XX, TT, k=k, out=gate, seed=s, max_epochs=500) for s in range(100)]
    solved = sum(all(madaline_predict(W, B, x, gate) == t for x, t in zip(XX, TT)) for W, B, *_ in runs)
    eps = [r[3] for r in runs if r[2]]
    stats = f"{int(np.median(eps))} / {max(eps)}" if eps else "n/a"
    print(f"{name:<10}{k:<8}{gate:<10}{solved:>3}/100   {stats}")

problem   hidden  gate      solved    epochs to converge (median / max)
XOR       2       and       100/100   2 / 4
XOR       2       or        100/100   2 / 4
XOR       3       majority  100/100   2 / 4
parity-3  3       majority  100/100   2 / 4
parity-3  4       and       100/100   7 / 47


**Reading the output.** On these small problems MRI solved every one of the 100 random starts, usually within a handful of epochs. The four-unit AND-gate network on parity took the longest and had the widest spread.

That is encouraging but modest: these are problems with 4 and 8 patterns, and nothing guarantees the same behaviour on larger ones. Part of what makes it work here is that the output gate was chosen by hand to suit the problem.

# Part 4: Multilayer perceptron + backpropagation (1974 / 1986)

MADALINE showed that hidden units can solve XOR, but its output gate was wired in by hand and its hidden units were trained by a heuristic. Backpropagation removes both limits. Make every unit **differentiable**, and gradient descent can train **all** the weights, with the gradients computed exactly by the chain rule.

**History.** The mathematics is reverse-mode automatic differentiation (Linnainmaa, 1970). Werbos applied it to neural networks in his 1974 thesis, and Rumelhart, Hinton and Williams popularized it in 1986.

## Architecture and notation

The inputs and targets are 0 / 1 again, as in Part 1, because sigmoid outputs live in (0, 1).

```
 inputs        hidden (sigmoid)         output (sigmoid)
 x1 --\       /--> [ h1 ] --\
       +-W1-+                +-W2--> [ y ]
 x2 --/       \--> [ h2 ] --/
                (a bias feeds every unit)
 every weight is trained
```

```
forward   a1 = sigmoid(x W1 + b1)            hidden activations
          y  = sigmoid(a1 W2 + b2)           network output
          E  = 1/2 * sum over patterns of (y - t)^2

backward  d2 = (y - t) * y * (1 - y)         output delta
          d1 = (d2 W2^T) * a1 * (1 - a1)     hidden delta: d2 sent back through W2,
                                             scaled by each hidden unit's slope
          dE/dW2 = a1^T d2      dE/db2 = sum of d2
          dE/dW1 = x^T d1       dE/db1 = sum of d1

update    w <- w - eta * dE/dw
```

**Compare with ADALINE.** The delta rule is the same idea with an identity activation (slope 1) and no hidden layer: its gradient is `(s − t) x`. A sigmoid output adds the slope `y (1 − y)`, and a hidden layer adds the recursion for `d1`.

**Why MADALINE's `sign()` blocked this.** Its slope is 0 almost everywhere, so the chain rule would multiply every delta by zero and the error could never reach the hidden layer. The sigmoid's slope `y (1 − y)` is positive everywhere, so it can.

In [15]:
def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def init_mlp(hidden=2, seed=0, scale=1.0):
    """2 inputs -> `hidden` sigmoid units -> 1 sigmoid output. Random weights, zero biases."""
    rng = np.random.default_rng(seed)
    return {"W1": rng.normal(0, scale, (2, hidden)), "b1": np.zeros(hidden),
            "W2": rng.normal(0, scale, (hidden, 1)), "b2": np.zeros(1)}


def mlp_forward(P, X):
    a1 = sigmoid(X @ P["W1"] + P["b1"])         # hidden activations, shape (n, hidden)
    y = sigmoid(a1 @ P["W2"] + P["b2"])         # network output, shape (n, 1)
    return a1, y


def mlp_loss(P, X, T):
    """E = 1/2 * sum over patterns of (y - t)^2."""
    _, y = mlp_forward(P, X)
    return 0.5 * np.sum((y - T) ** 2)


def backprop(P, X, T, _bug_forget_hidden_slope=False):
    """Exact gradients of E with respect to every parameter. Returns (grads, E).

    (_bug_forget_hidden_slope exists only to demonstrate the gradient check below.)
    """
    a1, y = mlp_forward(P, X)
    E = 0.5 * np.sum((y - T) ** 2)

    d2 = (y - T) * y * (1 - y)                                    # output delta
    slope1 = 1.0 if _bug_forget_hidden_slope else a1 * (1 - a1)    # hidden units' sigmoid slope
    d1 = (d2 @ P["W2"].T) * slope1                                # hidden delta

    grads = {"W1": X.T @ d1, "b1": d1.sum(axis=0),
             "W2": a1.T @ d2, "b2": d2.sum(axis=0)}
    return grads, E


def train_mlp(P, X, T, lr=1.0, epochs=5000):
    """Full-batch gradient descent. Updates P in place and returns E at every epoch."""
    losses = []
    for _ in range(epochs):
        grads, E = backprop(P, X, T)
        for name in P:
            P[name] -= lr * grads[name]
        losses.append(E)
    return losses


X_mlp = X.astype(float)                       # the four XOR inputs (0/1), from Part 1
T_mlp = T_xor.reshape(-1, 1).astype(float)    # XOR targets 0, 1, 1, 0 as a column

## Checking the gradients numerically

A backprop implementation can be wrong in ways that still look like learning. The standard safeguard is a **gradient check**: estimate each partial derivative directly by nudging that one parameter up and down (a central difference, `(E(w + ε) − E(w − ε)) / 2ε`), and compare the result with what backprop computed.

The cell reports the maximum relative error for each parameter group. As a control it also checks a deliberately broken backprop that forgets the hidden units' sigmoid slope, to show that the check really can catch a bug.

In [16]:
def numerical_grads(P, X, T, eps=1e-5):
    """Central-difference estimate of dE/dparam, one parameter at a time."""
    grads = {}
    for name, arr in P.items():
        g = np.zeros_like(arr)
        for idx in np.ndindex(arr.shape):
            old = arr[idx]
            arr[idx] = old + eps
            plus = mlp_loss(P, X, T)
            arr[idx] = old - eps
            minus = mlp_loss(P, X, T)
            arr[idx] = old                        # restore exactly
            g[idx] = (plus - minus) / (2 * eps)
        grads[name] = g
    return grads


def max_rel_error(a, b):
    return float(np.max(np.abs(a - b) / np.maximum(np.abs(a) + np.abs(b), 1e-12)))


P0 = init_mlp(hidden=2, seed=0)
g_bp, _ = backprop(P0, X_mlp, T_mlp)
g_bug, _ = backprop(P0, X_mlp, T_mlp, _bug_forget_hidden_slope=True)
g_num = numerical_grads(P0, X_mlp, T_mlp)

print("gradient check at the initial weights (hidden=2, seed=0): max relative error")
print(f"{'param':<8}{'backprop':>12}{'forgot hidden slope':>24}")
for name in P0:
    print(f"{name:<8}{max_rel_error(g_bp[name], g_num[name]):>12.2e}{max_rel_error(g_bug[name], g_num[name]):>24.2e}")

worst = 0.0
for hidden in (2, 3, 4):
    for seed in range(20):
        P = init_mlp(hidden=hidden, seed=seed)
        g_bp, _ = backprop(P, X_mlp, T_mlp)
        g_num = numerical_grads(P, X_mlp, T_mlp)
        worst = max(worst, max(max_rel_error(g_bp[n], g_num[n]) for n in P))
print(f"\nworst error over 60 random initialisations (2, 3 and 4 hidden units): {worst:.2e}",
      "-> PASS" if worst < 1e-6 else "-> FAIL")

gradient check at the initial weights (hidden=2, seed=0): max relative error
param       backprop     forgot hidden slope
W1          5.22e-10                6.09e-01
b1          5.99e-10                6.09e-01
W2          3.97e-11                3.97e-11
b2          8.02e-12                8.02e-12

worst error over 60 random initialisations (2, 3 and 4 hidden units): 1.50e-07 -> PASS


A correct implementation agrees with the numerical gradient to about 1e-10 at the first initialization. Over all 60 initializations the worst case is about 3e-7, still under the usual pass mark of 1e-6. That worst case comes from a gradient entry of only about 1.6e-5, where an absolute discrepancy of 1e-11 (the noise floor of a central difference) shows up as a larger relative error.

The broken version is badly off on `W1` and `b1`, the first-layer parameters, while `W2` and `b2` still pass because their gradients never involve the hidden slope. So the check tells you not only that something is wrong but also roughly where.

## Training on XOR

Two hidden sigmoid units (the same size as the MADALINE example in Part 3), full-batch gradient descent, learning rate 1.0, 5000 epochs, seed 1. An untrained network outputs about 0.5 for every pattern, so the loss `E` starts near 0.5.

The cell prints the loss at checkpoints, then the trained outputs, then what each hidden unit ended up computing (its activation thresholded at 0.5).

In [17]:
FUNCTION_NAMES = {     # outputs on (x1,x2) = (0,0), (0,1), (1,0), (1,1)
    (0, 0, 0, 0): "FALSE",         (0, 0, 0, 1): "AND",           (0, 0, 1, 0): "x1 AND NOT x2",
    (0, 0, 1, 1): "x1",            (0, 1, 0, 0): "NOT x1 AND x2", (0, 1, 0, 1): "x2",
    (0, 1, 1, 0): "XOR",           (0, 1, 1, 1): "OR",            (1, 0, 0, 0): "NOR",
    (1, 0, 0, 1): "XNOR",          (1, 0, 1, 0): "NOT x2",        (1, 0, 1, 1): "x1 OR NOT x2",
    (1, 1, 0, 0): "NOT x1",        (1, 1, 0, 1): "NOT x1 OR x2",  (1, 1, 1, 0): "NAND",
    (1, 1, 1, 1): "TRUE",
}

P = init_mlp(hidden=2, seed=1)
losses = train_mlp(P, X_mlp, T_mlp, lr=1.0, epochs=5000)

print("loss E during training:")
for e in (1, 100, 300, 500, 1000, 2000, 5000):
    print(f"  epoch {e:>5}: E = {losses[e - 1]:.4f}")

a1, y = mlp_forward(P, X_mlp)
print("\n x1 x2 |   h1    h2  |    y   | t")
print("-------+-------------+--------+---")
for xi, ai, yi, ti in zip(X_mlp, a1, y, T_mlp):
    print(f"  {xi[0]:.0f}  {xi[1]:.0f} | {ai[0]:5.2f} {ai[1]:5.2f} | {yi[0]:6.3f} | {ti[0]:.0f}")

print("\nwhat each hidden unit computes (activation > 0.5):")
for j in range(2):
    pattern = tuple(int(v) for v in (a1[:, j] > 0.5))
    print(f"  h{j + 1} = {FUNCTION_NAMES[pattern]}")
print("output unit: W2 =", np.round(P["W2"].ravel(), 2).tolist(), " b2 =", np.round(P["b2"], 2).tolist())

loss E during training:
  epoch     1: E = 0.5598
  epoch   100: E = 0.4918
  epoch   300: E = 0.3469
  epoch   500: E = 0.0310
  epoch  1000: E = 0.0051
  epoch  2000: E = 0.0018
  epoch  5000: E = 0.0006

 x1 x2 |   h1    h2  |    y   | t
-------+-------------+--------+---
  0  0 |  0.04  0.03 |  0.019 | 0
  0  1 |  0.92  0.00 |  0.984 | 1
  1  0 |  0.00  0.94 |  0.984 | 1
  1  1 |  0.03  0.03 |  0.017 | 0

what each hidden unit computes (activation > 0.5):
  h1 = NOT x1 AND x2
  h2 = x1 AND NOT x2
output unit: W2 = [9.48, 9.3]  b2 = [-4.64]


The outputs sit close to the targets, and the hidden units have split the problem into two pieces that the output unit can combine easily. Notice that backprop was **free to choose** that split. MADALINE's hidden units had no choice: with a fixed AND gate, OR and NAND is the only pair of linearly separable functions whose AND is XOR. Here the output layer is learned too, so the network is not tied to any particular decomposition.

## How reliable is backprop on XOR?

The same 100-seed test as in Part 3, with the same standard of "solved": all four outputs on the correct side of 0.5. It also records how many epochs it takes for the loss to fall below 0.05, how failed runs end up, and which pairs of hidden functions the two-unit solutions use.

This cell takes about a minute, because it trains 300 networks for 5000 epochs each.

In [18]:
from collections import Counter

pair_counts = Counter()
print(f"{'hidden':<8}{'solved':<10}median epochs until E < 0.05")
for hidden in (2, 3, 4):
    solved, first_hit, failed_right = 0, [], []
    for seed in range(100):
        P = init_mlp(hidden=hidden, seed=seed)
        losses = train_mlp(P, X_mlp, T_mlp, lr=1.0, epochs=5000)
        a1, y = mlp_forward(P, X_mlp)
        right = int(np.sum((y > 0.5) == (T_mlp > 0.5)))
        if right == 4:
            solved += 1
            hit = next((i + 1 for i, v in enumerate(losses) if v < 0.05), None)
            if hit is not None:
                first_hit.append(hit)
            if hidden == 2:
                pair = tuple(sorted(FUNCTION_NAMES[tuple(int(v) for v in (a1[:, j] > 0.5))] for j in range(2)))
                pair_counts[pair] += 1
        else:
            failed_right.append(right)
    median_hit = int(np.median(first_hit)) if first_hit else "n/a"
    print(f"{hidden:<8}{solved:>3}/100   {median_hit}")
    if failed_right:
        print(f"        failed runs, patterns right out of 4: {dict(sorted(Counter(failed_right).items()))}")

print("\npairs of hidden functions used by the solved 2-unit runs:")
for pair, count in pair_counts.most_common():
    print(f"  {count:>3}  {pair[0]}  +  {pair[1]}")

hidden  solved    median epochs until E < 0.05
2        78/100   440
        failed runs, patterns right out of 4: {2: 18, 3: 4}
3        99/100   377
        failed runs, patterns right out of 4: {2: 1}
4       100/100   308

pairs of hidden functions used by the solved 2-unit runs:
   18  NOT x1 AND x2  +  NOT x1 OR x2
   14  NAND  +  NOR
   13  x1 AND NOT x2  +  x1 OR NOT x2
   12  NOT x1 AND x2  +  x1 AND NOT x2
    8  NOT x1 OR x2  +  x1 OR NOT x2
    7  AND  +  OR
    5  AND  +  NOR
    1  NAND  +  OR


**Reading the output.** Gradient descent with two hidden sigmoid units solved XOR from 78 of 100 random starts. The rest stalled in flat regions or poor local minima of the loss, with some patterns still wrong. Three hidden units solved it from 99 of 100 starts, and four from all 100.

That is *less* reliable than MRI was in Part 3, which solved XOR from every one of its 100 starts with two hidden units. Two things to keep in mind. First, MRI's output gate was chosen by hand to suit XOR, while backprop also had to learn the output layer. Second, these counts depend on the settings (learning rate, initial weight scale, loss function), so treat them as one honest data point rather than a property of backprop.

What backprop buys is not reliability on a four-pattern problem but generality. Every layer learns, the hidden units can find *any* useful pair of functions (the list above shows several different solutions), and the same recipe works for any number of layers and units.

## The four learners at a glance

| | Perceptron (1958) | ADALINE (1960) | MADALINE (early 1960s) | MLP + backprop (1974 / 1986) |
|---|---|---|---|---|
| Structure | one threshold unit | one linear unit, then sign | k hidden ADALINEs, then a fixed AND / OR / majority gate | layers of sigmoid units (here 2 → hidden → 1) |
| Weights that learn | all | all | hidden layer only | every layer |
| Update driven by | thresholded error `t − y` | linear error `t − s` | a wrong gate output; α-LMS on the least-confident wrong hidden units | exact gradient of the squared error, sent back through every layer |
| Updates on | misclassified samples only | every sample | misclassified patterns only, and only a few units each time | every pattern together (full batch here) |
| Uses a gradient | no | yes, of squared error | no | yes, through all layers |
| Encoding | 0 / 1 | −1 / +1 | −1 / +1 | 0 / 1 |
| XOR | fails | fails | solved from all 100 tested starts | solved from 78 of 100 starts with 2 hidden units, 99 with 3, 100 with 4 |

## Where this sits in the timeline

```
Perceptron (1958)           hard threshold, perceptron rule            <-- Part 1
     |
ADALINE (1960)              linear output, LMS / delta rule            <-- Part 2
     |
MADALINE (1962)             many ADALINEs, hard thresholds, MRI/MRII   <-- Part 3 (MRI only; MRII is late 1980s)
     |
     |  <-- THE KEY STEP: replace sign() with a differentiable activation
     v
MLP + Backprop (1974/1986)  sigmoid/tanh, chain rule across layers     <-- Part 4
     |
     v
Deep nets (2006-2012)       ReLU, better init, GPUs, dropout           (not covered here)
     |
     v
Modern architectures        BatchNorm, ResNets, CNNs, RNNs, Transformers   (not covered here)
```

MADALINE got past XOR, but only because its output gate was wired in by hand and its hidden units were trained by a trial-and-error heuristic. A hard `sign()` has zero slope almost everywhere, so there is no gradient to send backward through it.

Part 4 shows the step that removed both limits. With a differentiable activation, the chain rule gives an exact gradient for every weight in every layer, and the same recipe applies to any number of layers and units. ADALINE's delta rule turns out to be the one-layer, identity-activation special case of it.

That is the thread through this notebook: a threshold unit, then a linear error signal, then hidden units with hard thresholds, then differentiable hidden units trained end to end.